In [4]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Nạp API key từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Khởi tạo Client với SDK mới
client = genai.Client(api_key=api_key)

try:
    # Sử dụng model gemini-3.8-flash theo khuyến nghị của Google
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Nói 'Xin chào!' bằng tiếng anh.",

    )
    print("Kết quả:", response.text)
except Exception as e:
    print("Lỗi khi gọi API:", e)

Kết quả: Bạn có thể nói là: **"Hello!"** hoặc **"Hi!"** nhé.


In [5]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Hãy tóm tắt tác dụng của Docker trong 2 gạch đầu dòng.",
    config=types.GenerateContentConfig(
        temperature=0.2,          # Logic, chính xác, ít lan man 0.0 nghiem tuc => 1.0 bay bong 
        max_output_tokens=100,    # Giới hạn độ dài để tiết kiệm hạn mức token
    )
)

print(response.text)

Dưới đây là tóm tắt tác dụng của Docker trong 2 gạch đầu dòng:

* **Đảm bảo tính nhất quán:** Đóng gói ứng dụng cùng toàn bộ môi trường (thư viện, cấu hình) vào các "container", giúp ứng dụng chạy ổn định trên mọi máy tính mà không gặp lỗi "trên máy tôi thì chạy được".
* **Tối ưu hóa triển khai:** Giúp việc cài đặt, di chuyển và mở rộng


Markdown
# Ghi nhớ: Kỹ thuật Chain-of-Thought (CoT) Prompting

## 1. Bản chất & Cơ chế
* **Định nghĩa:** Bắt LLM trình bày từng bước suy luận trung gian (dàn giáo suy nghĩ) trước khi chốt đáp án cuối cùng, thay vì nhảy cóc ra kết quả ngay.
* **Cơ chế:** LLM sinh chữ bằng cách dự đoán token tiếp theo. Khi viết ra bước 1, chính bước 1 trở thành ngữ cảnh đầu vào giúp mô hình tính toán bước 2 chuẩn xác hơn.
* **Bản chất:** Không nạp thêm kiến thức mới, chỉ "mở khóa" khả năng suy luận logic sẵn có của mô hình.

---

## 2. Hai hình thức triển khai

| Tiêu chí | Zero-shot CoT | Few-shot CoT |
| :--- | :--- | :--- |
| **Bản chất** | Dùng câu "thần chú", không cần ví dụ | Đưa kèm 1–2 bài toán mẫu đã giải sẵn |
| **Cách dùng** | Thêm: *"Hãy suy nghĩ từng bước một rồi đưa ra đáp án."* (*"Let's think step by step."*) | Đưa cặp mẫu `(User: bài toán -> Assistant: các bước giải + đáp án)` trước câu hỏi thật |
| **Ưu điểm** | Nhanh, ngắn gọn, tiết kiệm token | Ép khuôn định dạng và phong cách suy luận chính xác theo ý muốn |
| **Khi nào dùng**| Bài toán logic phổ thông, kiểm tra nhanh | Bài toán đặc thù, cấu trúc phức tạp, cần chuẩn hóa output |

In [6]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Nạp API key từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

# 2. Đề bài toán logic nhiều bước
bai_toan = (
    "Một lớp có 25 học sinh. 3/5 số đó là nữ. "
    "Trong số nữ, 4 bạn đeo kính. "
    "Hỏi có bao nhiêu bạn nữ KHÔNG đeo kính?"
)

# 3. Kỹ thuật Zero-shot CoT: Ghép câu 'thần chú' vào cuối đề bài
prompt_cot = f"{bai_toan}\n\nHãy suy nghĩ từng bước một, rồi đưa ra đáp án cuối cùng."

# 4. Cấu hình system instruction và temperature=0 cho logic ổn định
config = types.GenerateContentConfig(
    system_instruction="Bạn là trợ giảng Toán, luôn trình bày lời giải cẩn thận và rõ ràng từng bước.",
    temperature=0.0,         # Đặt 0 để suy luận logic ổn định, tránh bịa số
    max_output_tokens=600,   # Giới hạn token để tiết kiệm quota
)

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",  # Hoặc "gemini-3.8-flash"
        contents=prompt_cot,
        config=config,
    )
    print("--- LỜI GIẢI CỦA GEMINI ---")
    print(response.text)
except Exception as e:
    print("Lỗi:", e)

--- LỜI GIẢI CỦA GEMINI ---
Chào bạn, mình là trợ giảng Toán của bạn. Chúng ta sẽ cùng giải bài toán này theo từng bước nhé:

**Bước 1: Tính số học sinh nữ trong lớp.**
Theo đề bài, lớp có 25 học sinh và số học sinh nữ chiếm $\frac{3}{5}$ tổng số học sinh.
Số học sinh nữ là:
$25 \times \frac{3}{5} = 15$ (học sinh)

**Bước 2: Tính số học sinh nữ không đeo kính.**
Chúng ta biết rằng trong số 15 bạn nữ này, có 4 bạn đeo kính. Để tìm số bạn nữ không đeo kính, ta lấy tổng số bạn nữ trừ đi số bạn nữ có đeo kính:
$15 - 4 = 11$ (học sinh)

**Đáp số:** Có **11** bạn nữ không đeo kính.


In [7]:
prompt_few_shot = """Bạn là trợ lý giải toán logic, luôn phân tích theo các bước đánh số rõ ràng.

Ví dụ mẫu:
Câu hỏi: Có 12 quả táo, chia đều vào 3 rổ, sau đó bỏ bớt 1 rổ đi. Hỏi còn lại bao nhiêu quả táo?
Lời giải:
Bước 1: 12 chia 3 = 4 quả mỗi rổ.
Bước 2: Bỏ bớt 1 rổ tức là bớt đi 4 quả.
Bước 3: Số táo còn lại là 12 - 4 = 8 quả.
Đáp án: 8 quả.

---
Bây giờ hãy giải bài toán sau:
Câu hỏi: Một cửa hàng giảm giá 20%, sau đó giảm tiếp 10% trên giá đã giảm. Hỏi tổng cộng giảm bao nhiêu phần trăm so với giá gốc?
Lời giải:
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt_few_shot,
    config=types.GenerateContentConfig(temperature=0.0),
)

print(response.text)

Lời giải:

Bước 1: Giả sử giá gốc của sản phẩm là 100%. Sau khi giảm giá 20%, giá của sản phẩm còn lại là: 100% - 20% = 80% (giá gốc).

Bước 2: Tính mức giảm tiếp theo 10% trên giá đã giảm (80%): 10% × 80% = 8%.

Bước 3: Giá cuối cùng của sản phẩm là: 80% - 8% = 72% (giá gốc).

Bước 4: Tổng cộng phần trăm đã giảm so với giá gốc là: 100% - 72% = 28%.

Đáp án: 28%.


# Ghi nhớ: LangChain & LLM Chain Cơ Bản với LCEL

## 1. LangChain là gì?
* **Bản chất:** LangChain không phải mô hình AI thay thế LLM mà là **lớp keo (framework)** giúp ghép nối LLM với Prompt, Output Parser, Tool và Memory thành một dây chuyền làm việc mượt mà.
* **Vấn đề giải quyết:** Thay vì phải nối chuỗi `f"..."` thủ công, tự gọi API và tự bóc tách kết quả từ dict/JSON thô, LangChain chuẩn hóa mọi công đoạn thành các khối lắp ghép.

---

## 2. Ba thành phần lõi (Khuôn mẫu P-L-P)

| Thành phần | Class trong Code | Vai trò |
| :--- | :--- | :--- |
| **Prompt** | `ChatPromptTemplate` | Khuôn mẫu câu hỏi có chừa biến `{bien}` để tái sử dụng linh hoạt nhiều lần |
| **LLM** | `ChatOpenAI` *(hoặc `ChatGoogleGenerativeAI`)* | "Bộ não" nhận prompt và sinh phản hồi |
| **Parser** | `StrOutputParser` | "Người dọn dẹp" bóc tách tin nhắn thô thành chuỗi text thuần túy sạch sẽ |

---

## 3. Cú pháp LCEL (LangChain Expression Language)

* **Toán tử Pipe `|`:** Nối các thành phần theo dòng chảy dữ liệu từ trái sang phải:
  $$\text{Prompt (điền biến)} \longrightarrow \text{LLM (suy nghĩ/trả lời)} \longrightarrow \text{Parser (bóc tách chuỗi)}$$
* **Khuôn mẫu vàng:**
  ```python
  chain = prompt | llm | parser
  ket_qua = chain.invoke({"ten_bien": "gia_tri"})

In [8]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Nạp GOOGLE_API_KEY từ .env
load_dotenv()

# 1. Khuôn mẫu Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý dịch thuật, dịch sang tiếng Anh, chỉ trả về bản dịch."),
    ("human", "Dịch câu này: {cau}")
])

# 2. Khởi tạo LLM Gemini thay cho ChatOpenAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # Hoặc "gemini-3.8-flash"
    temperature=0.0
)

# 3. Bộ dọn dẹp output
parser = StrOutputParser()

# 4. Nối dây chuyền bằng LCEL (Pipe |)
chain = prompt | llm | parser

# 5. Chạy chain
ket_qua = chain.invoke({"cau": "Tôi yêu lập trình"})
print(ket_qua)
# Output: I love programming

I love programming.


# Ghi nhớ: Cơ chế Memory Hiện đại (Modern Memory) trong LangChain

## 1. Bản chất cốt lõi: LLM là Stateless
* **Stateless (Không trạng thái):** Mỗi lời gọi API (`invoke()`, `generate_content()`, `create()`) là một HTTP request độc lập. Server mô hình xóa sạch ngữ cảnh ngay sau khi trả lời.
* **Bản chất của Memory:** Mô hình không tự nhớ. "Trí nhớ" thực chất là quy trình lặp ở tầng ứng dụng:
  $$\text{Đọc lịch sử cũ} \longrightarrow \text{Nhét vào prompt} \longrightarrow \text{LLM trả lời} \longrightarrow \text{Ghi (input, output) mới vào sổ}$$

---

## 2. Classic Memory (Deprecated) vs Modern Approach

| Tiêu chí | Classic Memory (Đã khai tử) | Modern Memory (Hiện đại đang dùng) |
| :--- | :--- | :--- |
| **Các class đại diện** | `ConversationBufferMemory`, `ConversationSummaryMemory`, `ConversationChain` | `MessagesPlaceholder`, `RunnableWithMessageHistory`, `LangGraph State` |
| **Trạng thái hỗ trợ** | Bị **deprecated** từ LangChain 0.3.x | Chuẩn khuyến nghị chính thức (Standard) |
| **Kiến trúc luồng** | Ôm đồm, buộc dùng với `ConversationChain` cũ | Tách bạch giữa **Lưu trữ (Store)** và **Dây chuyền LCEL (`prompt \| llm`)** |
| **Hỗ trợ đa người dùng** | Khó cô lập nhiều phiên chat cùng lúc | Tự nhiên và chuẩn hóa qua tham số `session_id` |
| **Khả năng mở rộng** | Chủ yếu giữ trong RAM | Dễ dàng gắn cắm (pluggable) với Redis, MongoDB, Postgres |

---

## 3. Ba thành phần cấu tạo Modern Memory

1. **`MessagesPlaceholder(variable_name="history")`:** Một vị trí trống cắm sẵn bên trong `ChatPromptTemplate` để động nạp danh sách tin nhắn cũ vào prompt.
2. **Hàm `get_session_history(session_id)`:** Quản lý kho lưu trữ (`store`), trả về đối tượng `ChatMessageHistory` tương ứng với từng người dùng cụ thể.
3. **`RunnableWithMessageHistory`:** Wrapper bọc ngoài LCEL chain, tự động đọc lịch sử từ kho đưa vào placeholder và tự động ghi câu trả lời mới vào kho sau khi chạy xong.

---

In [9]:
import os
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser  # 1. Import Parser
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.0
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý thân thiện, trả lời ngắn gọn."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 2. Gắn parser vào chuỗi LCEL: prompt | llm | parser
chain = prompt | llm | StrOutputParser()

store = {}

def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. Bọc chain có parser
chat = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)

cfg = {"configurable": {"session_id": "quyen-01"}}

# Lượt 1: Kết quả trả về trực tiếp là string, không cần .content
res1 = chat.invoke({"input": "Tớ tên Quyên"}, config=cfg)
print("Lượt 1:", res1)

# Lượt 2:
res2 = chat.invoke({"input": "Tớ tên gì?"}, config=cfg)
print("Lượt 2:", res2)

C:\Users\kedie\AppData\Local\Temp\ipykernel_24164\2248166579.py:29: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


Lượt 1: Chào Quyên! Rất vui được làm quen với bạn. Mình có thể giúp gì cho bạn hôm nay không?
Lượt 2: Bạn tên là Quyên!


# Ghi nhớ: Cơ chế ReAct (Reasoning + Acting) trong AI Agent

## 1. Bản chất cốt lõi: ReAct là gì?
* **Khái niệm:** **ReAct** (*Reason + Act*) là mẫu thiết kế (design pattern) kết hợp giữa năng lực **suy luận** (*Reasoning*) và **hành động** (*Acting*).
* **Vấn đề giải quyết:**
  * LLM đơn thuần chỉ có *kiến thức tĩnh* (*static knowledge*), dễ bịa đặt thông tin (*hallucination*) khi gặp dữ kiện mới hoặc tính toán phức tạp.
  * ReAct trao cho LLM "đôi tay" (Tools) để tương tác với thế giới bên ngoài (tra cứu Wikipedia, tính toán, gọi API).
* **Khác biệt Agent vs Chain:**
  * **Chain:** Luồng chạy cố định, các bước do lập trình viên định sẵn (dây chuyền cố định).
  * **Agent:** LLM tự làm "đầu bếp" – tự quyết định khi nào cần dùng tool, dùng tool nào và lặp lại bao nhiêu lần tùy tình huống.

---

## 2. Vòng lặp ReAct Loop (Thought $\rightarrow$ Action $\rightarrow$ Observation)

Mô hình hoạt động theo chu trình 3 nhịp khép kín:

```text
┌───────────────────────────────────────────────┐
│  1. Thought (Suy luận): Mình cần dữ liệu gì?  │
│        ↓                                      │
│  2. Action (Hành động): Gọi tool nào, input?  │
│        ↓                                      │
│  3. Observation (Quan sát): Đọc kết quả tool  │
│        ↓                                      │
│     Đủ chưa? ── [Chưa] ──→ (Lặp lại nhịp 1)   │
│        ↓ [Đủ rồi]                             │
│  4. Final Answer: Chốt câu trả lời cuối cùng  │
└───────────────────────────────────────────────┘

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# 1. Khởi tạo Tool Wikipedia
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500) # lay thong itn voi k =1 
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)  # api 
tools = [wiki_tool]

# 2. Bộ não: LLM (đặt temperature=0 để suy luận logic ổn định)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.0
)

# 3. Prompt ReAct cục bộ, không cần tải cấu hình từ xa
prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

# 4. Ráp ReAct Agent và bọc trong AgentExecutor
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,  # Bắt buộc bật True để quan sát vết suy luận ReAct
    handle_parsing_errors=True
)

# 5. Thực thi câu hỏi cần tra cứu
ket_qua = executor.invoke({"input": "Ada Lovelace nổi tiếng vì điều gì?"})
print("Đáp án cuối:", ket_qua["output"])




> Entering new AgentExecutor chain...
Question: Ada Lovelace nổi tiếng vì điều gì?
Thought: Tôi cần tìm hiểu về Ada Lovelace để biết lý do tại sao bà ấy nổi tiếng.
Action: wikipedia
Action Input: Ada LovelacePage: Ada Lovelace
Summary: Augusta Ada King, Countess of Lovelace (née Byron; 10 December 1815 – 27 November 1852), also known as Ada Lovelace, was an English mathematician and writer chiefly known for work on Charles Babbage's proposed mechanical general-purpose computer, the analytical engine. She was the first to recognise the machine had applications beyond pure calculation. Lovelace is often considered the first computer programmer.
Lovelace was the only legitimate child of poet Lord ByronTôi đã có đủ thông tin để trả lời câu hỏi.

Final Answer: Ada Lovelace nổi tiếng vì công trình nghiên cứu của bà về chiếc máy tính cơ học đa năng của Charles Babbage, được gọi là "Analytical Engine" (Máy phân tích). Bà là người đầu tiên nhận ra rằng chiếc máy này có những ứng dụng vượt xa 

# Ghi nhớ: Agent với Nhiều Tool (Calculator, Search, Database)

## 1. Bản chất: Tool dưới góc nhìn của LLM
* **Cơ chế chọn tool:** LLM **không đọc code bên trong hàm**. Nó chỉ đọc "thẻ giới thiệu" bằng ngôn ngữ tự nhiên để đoán và quyết định:
  * **Tên hàm (`name`):** Gợi ý chức năng chính (ví dụ: `calculator`, `query_database`).
  * **Tham số & kiểu dữ liệu (`args`):** Biết cần truyền dữ liệu gì vào.
  * **Docstring (`description`):** **Linh hồn của tool** — hướng dẫn trực tiếp cho LLM biết *KHI NÀO* nên dùng và *KHI NÀO KHÔNG* nên dùng.
* **Quy tắc vàng:** Docstring mơ hồ $\rightarrow$ Agent chọn nhầm tool. Khi debug Agent chọn sai công cụ, hãy **sửa docstring và tên hàm trước**, chưa cần sửa logic code.

---

## 2. Công thức viết Docstring chuẩn cho `@tool`
Một docstring tốt cần hội tụ đủ 3 yếu tố:
1. **Tool làm gì** (Chức năng cốt lõi).
2. **Dùng khi nào / Không dùng khi nào** (Phân định ranh giới trách nhiệm rõ ràng).
3. **Ví dụ đầu vào mẫu** (Format dữ liệu truyền vào).

In [18]:
from langchain_core.tools import tool

@tool
def calculator(bieu_thuc: str) -> str:
    """Tính giá trị biểu thức toán học.
    Dùng khi cần CỘNG, TRỪ, NHÂN, CHIA hoặc tính toán số học.
    Ví dụ đầu vào: '2035 * 47'."""
    return str(eval(bieu_thuc))

@tool
def search(tu_khoa: str) -> str:
    """Tra cứu dữ kiện thực tế công khai trên web (tin tức, định nghĩa, sự kiện).
    KHÔNG dùng để tra dữ liệu nội bộ công ty."""
    return f"Kết quả web cho: {tu_khoa}"

@tool
def query_database(cau_hoi: str) -> str:
    """Truy vấn dữ liệu NỘI BỘ của công ty (đơn hàng, khách hàng, doanh thu).
    Dùng khi câu hỏi liên quan số liệu nội bộ, KHÔNG dùng tra web."""
    return "Doanh thu tháng 6: 505 triệu"

In [22]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# 1. Định nghĩa các công cụ bằng @tool chuẩn LangChain
@tool
def calculator(bieu_thuc: str) -> str:
    """Tính giá trị biểu thức số học (+, -, *, /). 
    Dùng khi cần tính toán toán học. Ví dụ đầu vào: '2035 * 47'."""
    try:
        return str(eval(bieu_thuc))
    except Exception as e:
        return f"Lỗi tính toán: {e}"

@tool
def search(tu_khoa: str) -> str:
    """Tra cứu dữ kiện thực tế công khai trên web (tin tức, sự kiện, thủ đô, con người).
    KHÔNG dùng để tra cứu số liệu nội bộ công ty."""
    return f"Kết quả tra cứu web cho '{tu_khoa}': Thủ đô nước Pháp là Paris."

@tool
def query_database(cau_hoi: str) -> str:
    """Truy vấn dữ liệu NỘI BỘ của công ty (doanh thu, đơn hàng, khách hàng).
    Dùng khi câu hỏi liên quan đến số liệu công ty, KHÔNG dùng tra web."""
    return "Doanh thu tháng 6 của công ty là 505 triệu VNĐ."

tools = [calculator, search, query_database]

# 2. Khởi tạo LLM Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.0
)

# 3. Prompt chuẩn cho Tool Calling Agent (không cần pull từ LangChain Hub)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý AI hữu ích, có quyền truy cập vào các công cụ để trả lời chính xác."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),  # Nơi lưu vết suy nghĩ và kết quả gọi Tool
])

# 4. Ráp Agent bằng Tool Calling gốc (Native Tool Calling)
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5. Chạy thử nghiệm
print("\n--- Test 1: Tính toán ---")
res1 = executor.invoke({"input": "2035 nhân 47 bằng bao nhiêu?"})
print("Đáp án:", res1["output"])

print("\n--- Test 2: Dữ liệu công ty ---")
res2 = executor.invoke({"input": "Doanh thu tháng 6 của công ty là bao nhiêu?"})
print("Đáp án:", res2["output"])

print("\n--- Test 3: Kết hợp nhiều tool (Database -> Calculator) ---")
res3 = executor.invoke({"input": "Doanh thu tháng 6 của công ty chia đều cho 5 chi nhánh thì mỗi nơi được bao nhiêu triệu?"})
print("Đáp án:", res3["output"])


--- Test 1: Tính toán ---


> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'bieu_thuc': '2035 * 47'}`


95645[{'type': 'text', 'text': '2035 nhân 47 bằng 95.645.', 'index': 0, 'extras': {'signature': 'EnMKcQFpFH0T2+jDp3YtMmlI7YG22FPWPTK2HY/m5lfljmknl/2iYx73pzwatiXGu7rNZcSCTGsQX420ssnmGUXHTPykWFLjKJWYdWy/cLteJxH/3TZjun2OITEPbBDkjS4APNBtFSmpQhNkJw36r6IJ8YcD'}}]

> Finished chain.
Đáp án: [{'type': 'text', 'text': '2035 nhân 47 bằng 95.645.', 'index': 0, 'extras': {'signature': 'EnMKcQFpFH0T2+jDp3YtMmlI7YG22FPWPTK2HY/m5lfljmknl/2iYx73pzwatiXGu7rNZcSCTGsQX420ssnmGUXHTPykWFLjKJWYdWy/cLteJxH/3TZjun2OITEPbBDkjS4APNBtFSmpQhNkJw36r6IJ8YcD'}}]

--- Test 2: Dữ liệu công ty ---


> Entering new AgentExecutor chain...

Invoking: `query_database` with `{'cau_hoi': 'Doanh thu tháng 6 của công ty là bao nhiêu?'}`


Doanh thu tháng 6 của công ty là 505 triệu VNĐ.[{'type': 'text', 'text': 'Doanh thu tháng 6 của công ty là 505 triệu VNĐ.', 'index': 0, 'extras': {'signature': 'EnMKc

In [4]:
import os
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
# Cài đặt thư viện: pip install langchain-google-genai
# Thiết lập API key: os.environ["GOOGLE_API_KEY"] = "your-api-key"
load_dotenv()

# 1) Vài ví dụ mẫu đại diện cho 3 ý định
examples = [
    {"text": "Cho mình đặt 2 ly trà sữa trân châu", "intent": "đặt hàng"},
    {"text": "Đơn của tôi 5 ngày rồi chưa thấy giao", "intent": "khiếu nại"},
    {"text": "Cửa hàng mở cửa tới mấy giờ vậy shop?", "intent": "hỏi thông tin"},
    {"text": "Sản phẩm giao tới bị vỡ, tôi rất bực", "intent": "khiếu nại"},
]

# 2) Khuôn để in MỖI ví dụ
example_prompt = PromptTemplate(
    input_variables=["text", "intent"],
    template="Câu: {text}\nÝ định: {intent}",
)

# 3) Ghép thành prompt few-shot hoàn chỉnh
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Phân loại ý định của khách thành đúng MỘT nhãn: đặt hàng, khiếu nại, hoặc hỏi thông tin.",
    suffix="Câu: {input}\nÝ định:",
    input_variables=["input"],
)

# 4) Khởi tạo Gemini model thay cho ChatOpenAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
)

# 5) Chạy thử nghiệm
cau_moi = "Mình muốn mua thêm 1 phần bánh flan nữa"
chain = prompt | llm
ket_qua = chain.invoke({"input": cau_moi})

print(ket_qua.content.strip())  # -> đặt hàng

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [5]:
from langchain_community.llms import Ollama
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
# Hoặc dùng class mới: from langchain_ollama import ChatOllama

# 1) Tập ví dụ few-shot
examples = [
    {"text": "Cho mình đặt 2 ly trà sữa trân châu", "intent": "đặt hàng"},
    {"text": "Đơn của tôi 5 ngày rồi chưa thấy giao", "intent": "khiếu nại"},
    {"text": "Cửa hàng mở cửa tới mấy giờ vậy shop?", "intent": "hỏi thông tin"},
    {"text": "Sản phẩm giao tới bị vỡ, tôi rất bực", "intent": "khiếu nại"},
]

# 2) Prompt template cho từng ví dụ
example_prompt = PromptTemplate(
    input_variables=["text", "intent"],
    template="Câu: {text}\nÝ định: {intent}",
)

# 3) Ghép prompt hoàn chỉnh
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Phân loại ý định của khách thành đúng MỘT nhãn: đặt hàng, khiếu nại, hoặc hỏi thông tin.",
    suffix="Câu: {input}\nÝ định:",
    input_variables=["input"],
)

# 4) Khởi tạo model chạy local trên máy
llm = Ollama(
    model="gemma3:1b",  # hoặc gemma2:9b, llama3.1:8b
    temperature=0,
)

# 5) Chạy thử nghiệm
cau_moi = "Mình muốn mua thêm 1 phần bánh flan nữa"
chain = prompt | llm
ket_qua = chain.invoke({"input": cau_moi})

print(ket_qua.strip())  # -> đặt hàng

C:\Users\kedie\AppData\Local\Temp\ipykernel_11848\3523707540.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
C:\Users\kedie\AppData\Local\Temp\ipykernel_11848\3523707540.py:29: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


đặt hàng


In [7]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_ollama import OllamaLLM  # Dùng package mới thay cho langchain_community

# 1) Các ví dụ mẫu
examples = [
    {"text": "Cho mình đặt 2 ly trà sữa trân châu", "intent": "đặt hàng"},
    {"text": "Đơn của tôi 5 ngày rồi chưa thấy giao", "intent": "khiếu nại"},
    {"text": "Cửa hàng mở cửa tới mấy giờ vậy shop?", "intent": "hỏi thông tin"},
    {"text": "Sản phẩm giao tới bị vỡ, tôi rất bực", "intent": "khiếu nại"},
]

# 2) Prompt template
example_prompt = PromptTemplate(
    input_variables=["text", "intent"],
    template="Câu: {text}\nÝ định: {intent}",
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Phân loại ý định của khách thành đúng MỘT nhãn: đặt hàng, khiếu nại, hoặc hỏi thông tin.",
    suffix="Câu: {input}\nÝ định:",
    input_variables=["input"],
)

# 3) Khởi tạo model sạch warning
llm = OllamaLLM(
    model="gemma3:1b",
    temperature=0,
)

chain = prompt | llm
ket_qua = chain.invoke({"input": "Mình muốn mua thêm 1 phần bánh flan nữa"})
print(ket_qua.strip())

đặt hàng


In [11]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

# 1. Định nghĩa schema rõ ràng
class PhanTichDonHang(BaseModel):
    intent: str = Field(description="Ý định của khách, ví dụ: đặt hàng")
    mon_an: str = Field(description="Tên món ăn được nhắc đến")
    so_luong: int = Field(default=1, description="Số lượng")

# 2. Bọc model
llm = ChatOllama(model="gemma3:1b", temperature=0)
structured_llm = llm.with_structured_output(PhanTichDonHang)

# 3. Kết quả trả về là Object Pydantic chuẩn
ket_qua = structured_llm.invoke("Mình muốn mua thêm 1 phần bánh flan nữa")

# In ra text chuẩn JSON
print(ket_qua.model_dump_json(indent=2))

{
  "intent": "order_food",
  "mon_an": "flan",
  "so_luong": 1
}


# Tóm tắt: Agent ReAct (Reasoning + Acting)

### 1. Bản chất ReAct
* **ReAct = Reasoning + Acting:** Kết hợp giữa tư duy logic từng bước (*Chain-of-Thought*), gọi công cụ ngoài (*Tool Calling*) và ép khuôn dữ liệu (*Structured Output*).
* **Vòng lặp "T-A-O":** Hoạt động theo chu trình xen kẽ liên tục cho tới khi ra kết quả cuối:
  $$\text{Thought} \rightarrow \text{Action (kèm Action Input)} \rightarrow \text{Observation} \rightarrow \dots \rightarrow \text{Final Answer}$$

---

### 2. Vai trò trong vòng lặp
* **Thought (Model sinh):** Mô hình suy nghĩ bước tiếp theo cần làm gì.
* **Action & Action Input (Model sinh):** Tên công cụ và đối số truyền vào (ví dụ: `search("dân số VN")`).
* **Observation (Code Python thực thi):** Kết quả tra cứu thật do **chương trình chạy tool** rồi gắn ngược lại vào ngữ cảnh. Model tuyệt đối **không tự bịa** ra dòng này.
* **Final Answer (Model sinh):** Đưa ra đáp án cuối cùng và kết thúc tác vụ.

---

### 3. Ba điểm kỹ thuật cốt lõi khi triển khai
1. **Dùng Stop Sequences (`stop=["Observation:"]`):** Ép mô hình dừng bút ngay sau khi sinh ra `Action Input`, nhường quyền thực thi cho code điền dữ liệu thật.
2. **Chốt chặn số bước (`max_steps`):** Giới hạn số lần lặp (thường từ 6–10 bước) để tránh mô hình bị kẹt trong vòng lặp vô tận gây tốn token/chi phí.
3. **Parse chuẩn xác:** Chuẩn hóa chuỗi text hoặc ép kiểu sang JSON (`{"action": "...", "action_input": "..."}`) để code gọi đúng hàm mà không bị lỗi cú pháp thừa chữ.

In [13]:
import re
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

# 1. Khởi tạo model Ollama local chuẩn hiện nay
llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
    # Chặn model tự bịa dòng Observation (ngừng bút ngay sau khi sinh Action Input)
    stop=["Observation:"],
)

# 2. Định nghĩa danh sách các Tools
def search(query: str) -> str:
    # Bỏ dấu ngoặc kép thừa mà model hay sinh ra
    q = query.lower().replace('"', '').replace("'", "").strip()
    
    kho = {
        ("oscar 2020", "oscar năm 2020", "oscar"): "Parasite, dao dien Bong Joon-ho, thang Best Picture.",
        ("bong joon-ho", "bong joon ho", "đạo diễn parasite"): "Bong Joon-ho sinh nam 1969.",
        ("parasite", "ký sinh trùng"): "Parasite do Bong Joon-ho dao dien thang Oscar 2020.",
    }
    
    for keys, val in kho.items():
        if any(k in q for k in keys):
            return val
            
    return "Khong tim thay thong tin. Hay thu tim kiem bang tu khoa ngan gon hon (vi du: 'oscar 2020')."

def calculator(bieu_thuc: str) -> str:
    """Thực hiện tính toán biểu thức an toàn cơ bản"""
    try:
        # Lọc ký tự toán học cơ bản tránh rủi ro
        clean_expr = re.sub(r"[^0-9+\-*/(). ]", "", bieu_thuc)
        return str(eval(clean_expr))
    except Exception as e:
        return f"Loi tinh toan: {e}"

TOOLS = {
    "search": search,
    "calculator": calculator,
}

# 3. System Prompt ReAct theo đúng chuẩn bài học
SYSTEM_PROMPT = """Bạn giải bài toán theo vòng ReAct. Ở mỗi bước xuất đúng format:
Thought: (suy nghĩ)
Action: (chọn một trong: [search, calculator])
Action Input: (từ khóa ngắn gọn, không để trong ngoặc kép)

Lưu ý: Nếu tìm kiếm không thấy, hãy thử từ khóa ngắn hơn.
Khi đã đủ thông tin, xuất:
Final Answer: (câu trả lời)

Ví dụ:
Question: Ai là tác giả Truyện Kiều?
Thought: Cần tìm tác giả Truyện Kiều.
Action: search
Action Input: truyen kieu
"""

# 4. Hàm helper trích xuất an toàn dữ liệu từ output của LLM
def lay_gia_tri(text: str, nhan: str) -> str:
    for dong in text.splitlines():
        dong_strip = dong.strip()
        if dong_strip.startswith(nhan):
            return dong_strip[len(nhan):].strip()
    return ""

# 5. Vòng lặp điều phối ReAct (Loop Controller)
def run_react(question: str, max_steps: int = 6):
    print(f"=== Bắt đầu câu hỏi: {question} ===\n")
    
    # Khởi tạo scratchpad lịch sử
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Question: {question}"),
    ]
    
    for buoc in range(1, max_steps + 1):
        print(f"--- [Vòng {buoc}] ---")
        
        # Gọi model local
        response = llm.invoke(messages)
        content = response.content.strip()
        print(content)
        
        # 1. Kiểm tra điều kiện DỪNG
        if "Final Answer:" in content:
            final_ans = lay_gia_tri(content, "Final Answer:")
            print("\n===============================")
            print(f"-> KẾT QUẢ CUỐI CÙNG: {final_ans}")
            print("===============================")
            return final_ans

        # 2. Parse Action và Action Input
        action = lay_gia_tri(content, "Action:").lower().replace(".", "")
        arg = lay_gia_tri(content, "Action Input:")
        
        # 3. Code Python thực thi Tool thật để tạo Observation
        if action in TOOLS:
            observation = TOOLS[action](arg)
        else:
            observation = f"Lỗi: Không tìm thấy tool có tên '{action}'."
            
        print(f"Observation: {observation}\n")
        
        # 4. Ghép vòng lặp vào lịch sử hội thoại cho bước kế tiếp
        messages.append(AIMessage(content=content))
        messages.append(HumanMessage(content=f"Observation: {observation}"))
        
    print("Đã hết số vòng tối đa (max_steps) mà chưa có câu trả lời.")
    return None

# --- Chạy thử nghiệm ---
if __name__ == "__main__":
    cau_hoi = "Đạo diễn của bộ phim đoạt Oscar Phim hay nhất năm 2020 sinh năm nào?"
    run_react(cau_hoi)

=== Bắt đầu câu hỏi: Đạo diễn của bộ phim đoạt Oscar Phim hay nhất năm 2020 sinh năm nào? ===

--- [Vòng 1] ---
Thought: Cần tìm đạo diễn của bộ phim đoạt Oscar.
Action: search
Action Input: đạo diễn phim đoạt Oscar 2020
Observation: Parasite, dao dien Bong Joon-ho, thang Best Picture.

--- [Vòng 2] ---
Thought: Xác định đạo diễn của Parasite.
Action: search
Action Input: Bong Joon-ho Parasite best picture
Observation: Bong Joon-ho sinh nam 1969.

--- [Vòng 3] ---
Thought: Xác định tuổi sinh của Bong Joon-ho.
Action: search
Action Input: Bong Joon-ho sinh nam 1969
Observation: Bong Joon-ho sinh nam 1969.

--- [Vòng 4] ---
Thought: Xác định tuổi sinh của Bong Joon-ho.
Action: search
Action Input: Bong Joon-ho sinh nam 1969
Observation: Bong Joon-ho sinh nam 1969.

--- [Vòng 5] ---
Thought: Xác định tuổi sinh của Bong Joon-ho.
Action: search
Action Input: Bong Joon-ho sinh nam 1969
Observation: Bong Joon-ho sinh nam 1969.

--- [Vòng 6] ---
Thought: Xác định tuổi sinh của Bong Joon-ho.
Ac

In [16]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# 1. Định nghĩa Tools (Giữ nguyên đầu vào 1 chuỗi cho ReAct cổ điển)
@tool
def nhan_hai_so(bieu_thuc: str) -> str:
    """Nhân hai số. Đầu vào dạng 'a,b' ví dụ '6,7'. Trả về tích a*b."""
    clean_input = bieu_thuc.strip().replace("'", "").replace('"', "")
    a, b = clean_input.split(",")
    return str(int(a.strip()) * int(b.strip()))

@tool
def do_dai_chuoi(text: str) -> str:
    """Đếm số ký tự của một chuỗi. Đầu vào là chuỗi bất kỳ."""
    clean_text = text.strip().replace("'", "").replace('"', "")
    return str(len(clean_text))

tools = [nhan_hai_so, do_dai_chuoi]

# 2. Model Local
llm = ChatOllama(
    model="gemma3:1b",
    temperature=0,
)

# 3. Định nghĩa trực tiếp Prompt ReAct bằng tiếng Việt giúp model 1B hiểu tốt hơn
template = """Trả lời câu hỏi sau một cách tốt nhất có thể. Bạn có quyền truy cập vào các công cụ sau:

{tools}

Sử dụng định dạng sau:

Question: câu hỏi đầu vào bạn phải trả lời
Thought: bạn nên luôn suy nghĩ về việc cần làm gì
Action: hành động cần thực hiện, phải là một trong [{tool_names}]
Action Input: đầu vào cho hành động (không kèm dấu ngoặc kép thừa)
Observation: kết quả của hành động
... (lặp lại Thought/Action/Action Input/Observation nếu cần)
Thought: Tôi đã biết câu trả lời cuối cùng
Final Answer: câu trả lời cuối cùng cho câu hỏi ban đầu

Bắt đầu!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)

# 4. Tạo Agent & Executor
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

# 5. Chạy
ket_qua = agent_executor.invoke({
    "input": "6 nhân 7 bằng mấy, rồi cho biết độ dài của chữ LangChain?"
})

print("\n--- KẾT QUẢ CUỐI CÙNG ---")
print(ket_qua["output"])



> Entering new AgentExecutor chain...
Tôi cần thực hiện phép nhân 6 nhân 7. Sau đó, tôi sẽ tính độ dài của chuỗi "LangChain".
Action: nhan_hai_so(bieu_thuc: '6,7')
Action Input: 6,7nhan_hai_so(bieu_thuc: '6,7') is not a valid tool, try one of [nhan_hai_so, do_dai_chuoi].Tôi cần thực hiện phép nhân. Tôi sẽ sử dụng công cụ nhan_hai_so để thực hiện phép nhân 6 và 7.
Action: nhan_hai_so(bieu_thuc: '6,7')
Action Input: 6,7nhan_hai_so(bieu_thuc: '6,7') is not a valid tool, try one of [nhan_hai_so, do_dai_chuoi].Tôi cần thực hiện phép nhân. Tôi sẽ sử dụng công cụ nhan_hai_so để thực hiện phép nhân 6 và 7.
Action: nhan_hai_so(bieu_thuc: '6,7')
Action Input: 6,7nhan_hai_so(bieu_thuc: '6,7') is not a valid tool, try one of [nhan_hai_so, do_dai_chuoi].6 * 7 = 42
The length of the string "LangChain" is 14.
Final Answer: 42

> Finished chain.

--- KẾT QUẢ CUỐI CÙNG ---
42


In [19]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# 1. Định nghĩa Tools
@tool
def nhan_hai_so(a: int, b: int) -> int:
    """Nhân hai số nguyên a và b."""
    return a * b

@tool
def do_dai_chuoi(text: str) -> int:
    """Đếm số ký tự của một chuỗi bất kỳ."""
    return len(text)

tools = [nhan_hai_so, do_dai_chuoi]

# 2. Định nghĩa cấu trúc dữ liệu đầu ra (Structured Output)
class KetQuaTinhToan(BaseModel):
    tich_hai_so: int = Field(description="Kết quả của phép nhân hai số")
    so_ky_tu: int = Field(description="Độ dài (số ký tự) của chuỗi được yêu cầu")
    loi_giai_thich: str = Field(description="Câu trả lời tóm tắt ngắn gọn bằng tiếng Việt")

# 3. Khởi tạo Model Local
llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0,
)

# 4. Tạo Agent với ToolStrategy (Thay vì truyền trực tiếp class Pydantic)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "Bạn là trợ lý thông minh. Hãy gọi các công cụ tính toán trước, "
        "sau đó bắt buộc hoàn thành kết quả theo đúng cấu trúc yêu cầu."
    ),
    response_format=ToolStrategy(KetQuaTinhToan),
)

# 5. Chạy Agent
inputs = {
    "messages": [
        {"role": "user", "content": "6 nhân 7 bằng mấy, rồi cho biết độ dài của chữ LangChain?"}
    ]
}

result = agent.invoke(inputs, config={"recursion_limit": 10})

# 6. Lấy dữ liệu (Kèm cơ chế Fallback chống lỗi None tuyệt đối cho model 1.5B)
structured_data: KetQuaTinhToan = result.get("structured_response")

if structured_data is None:
    # Nếu model 1.5B chỉ trả lời bằng chữ mà quên đóng gói vào structured_response,
    # dùng .with_structured_output() ép trực tiếp từ tin nhắn cuối cùng:
    last_message_text = result["messages"][-1].content
    structured_llm = llm.with_structured_output(KetQuaTinhToan)
    structured_data = structured_llm.invoke(
        f"Trích xuất thông tin sau thành đúng định dạng JSON:\n{last_message_text}"
    )

print("--- ĐỐI TƯỢNG PYDANTIC ---")
print(structured_data)

print("\n--- TRUY XUẤT TỪNG TRƯỜNG DỮ LIỆU ---")
print(f"Tích 2 số: {structured_data.tich_hai_so}")
print(f"Độ dài chuỗi: {structured_data.so_ky_tu}")
print(f"Giải thích: {structured_data.loi_giai_thich}")

print("\n--- XUẤT RA JSON CHUẨN ---")
print(structured_data.model_dump_json(indent=2))

--- ĐỐI TƯỢNG PYDANTIC ---
tich_hai_so=6 so_ky_tu=7 loi_giai_thich='bytes'

--- TRUY XUẤT TỪNG TRƯỜNG DỮ LIỆU ---
Tích 2 số: 6
Độ dài chuỗi: 7
Giải thích: bytes

--- XUẤT RA JSON CHUẨN ---
{
  "tich_hai_so": 6,
  "so_ky_tu": 7,
  "loi_giai_thich": "bytes"
}


# Bài 11: Agent có khả năng lập kế hoạch (Plan-and-Execute)

> **Mô hình cốt lõi:** Tách "nghĩ" ra khỏi "làm"  
> `ReAct` = "Vừa đi vừa nghĩ từng bước"  
> `Plan-and-Execute` = "Vạch xong cả lộ trình rồi mới bước, lệch thì vẽ lại bản đồ"

---

## 1. Kiến trúc tổng quan (Mô hình P-E-R)

Kiến trúc Plan-and-Execute chia thành 3 vai trò rõ ràng:

1. **Planner (Người lập kế hoạch):** Nhận mục tiêu tổng quát, dùng model mạnh/thông minh để chia thành danh sách các bước cụ thể: `[B1, B2, B3, ...]`.
2. **Executor (Người thực thi):** Lấy từng bước một và thực hiện (thường bản thân Executor là một sub-agent chạy ReAct có tích hợp sẵn Tools).
3. **Replanner (Người cập nhật kế hoạch):** Quan sát kết quả đã thực hiện, quyết định cập nhật/thêm/bớt bước hoặc xác nhận hoàn thành mục tiêu.

```text
Mục tiêu người dùng
       │
       ▼
+--------------+
│   PLANNER    │ ──► [B1, B2, B3, B4] (Kế hoạch nhiều bước)
+--------------+
       │
       ▼
+--------------+      Lấy bước đầu
│   EXECUTOR   │ ◄─── Thực thi B1, trả kết quả
+--------------+
       │
       ▼
+--------------+      Còn bước? ──► Làm tiếp
│  REPLANNER   │ ──►  Lệch?     ──► Vẽ lại kế hoạch
+--------------+      Hết?      ──► Hoàn thành & Trả lời

In [5]:
import operator
from typing import Annotated, List, Tuple, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent  # Chuẩn mới từ LangChain/LangGraph V1.0+
from langgraph.graph import StateGraph, START, END
from langchain_core.globals import set_verbose, set_debug
# ==========================================
# 1. KHỞI TẠO MODEL OLLAMA & TOOLS
# ==========================================
llm = ChatOllama(model="qwen2.5:1.5b", temperature=0)
set_verbose(True)
@tool
def search_population(country: str) -> str:
    """Tra cứu dân số của một quốc gia (ví dụ: 'Việt Nam', 'Thái Lan'). Chỉ truyền tên 1 quốc gia mỗi lần gọi."""
    c = country.lower().strip()
    if any(k in c for k in ["việt nam", "vietnam", "viet nam", "vn"]):
        return "Dân số Việt Nam là 100.3 triệu người."
    if any(k in c for k in ["thái lan", "thailand", "thai lan"]):
        return "Dân số Thái Lan là 71.8 triệu người."
    if any(k in c for k in ["nhật bản", "japan", "nhat ban"]):
        return "Dân số Nhật Bản là 124.0 triệu người."
    return f"Không tìm thấy dữ liệu dân số cho '{country}'."

@tool
def calculate_diff(a: float, b: float) -> str:
    """Tính độ chênh lệch tuyệt đối giữa hai con số a và b (đơn vị triệu người)."""
    return f"Độ chênh lệch là {abs(a - b):.1f} triệu người."

tools = [search_population, calculate_diff]

# Sử dụng create_agent theo chuẩn mới
executor_agent = create_agent(llm, tools=tools)


# ==========================================
# 2. ĐỊNH NGHĨA STATE & STRUCTURED OUTPUT (PHẲNG)
# ==========================================
class PlanExecuteState(TypedDict):
    input: str
    plan: List[str]
    past_steps: Annotated[List[Tuple[str, str]], operator.add]
    response: str

class Plan(BaseModel):
    """Kế hoạch gồm danh sách các bước cần thực hiện."""
    steps: List[str] = Field(
        description="Danh sách các bước ngắn gọn. Mỗi bước chỉ làm 1 việc (ví dụ: Bước 1 tìm dân số nước A, Bước 2 tìm dân số nước B, Bước 3 tính chênh lệch)."
    )

# Làm phẳng Schema cho Replanner để model Ollama 7B không bị nhầm lẫn
class ReplanDecision(BaseModel):
    """Quyết định của Replanner sau khi kiểm tra kết quả các bước đã làm."""
    is_complete: bool = Field(
        description="Đặt là True CHỈ KHI đã có đầy đủ số liệu dân số của cả 2 nước và độ chênh lệch để trả lời người dùng. Nếu chưa đủ hoặc bước trước bị lỗi, đặt là False."
    )
    final_response: Optional[str] = Field(
        default=None,
        description="Nếu is_complete=True, viết câu trả lời tổng hợp đầy đủ số liệu tại đây."
    )
    remaining_steps: Optional[List[str]] = Field(
        default=None,
        description="Nếu is_complete=False, liệt kê danh sách CÁC BƯỚC CÒN LẠI cần tiếp tục thực hiện."
    )


# ==========================================
# 3. THIẾT LẬP PLANNER & REPLANNER CHAINS
# ==========================================
planner_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Bạn là chuyên gia lập kế hoạch. Hãy chia yêu cầu của người dùng thành từng bước nhỏ, tuần tự. "
     "Lưu ý: Tách riêng việc tra cứu từng quốc gia thành từng bước riêng biệt."),
    ("user", "{input}"),
])
planner = planner_prompt | llm.with_structured_output(Plan)

replanner_prompt = ChatPromptTemplate.from_template(
    """Bạn là người điều phối kế hoạch (Replanner). Hãy kiểm tra tiến độ thực hiện nhiệm vụ.

Mục tiêu ban đầu của người dùng:
{input}

Kế hoạch hiện tại:
{plan}

Các bước ĐÃ LÀM và KẾT QUẢ thu được:
{past_steps}

Quy tắc đánh giá:
1. Nhìn vào phần 'KẾT QUẢ thu được'. Nếu đã có đủ số liệu dân số từng nước và kết quả tính chênh lệch -> Đặt is_complete = True và viết câu trả lời chi tiết vào final_response.
2. Nếu chưa làm hết các bước trong kế hoạch -> Đặt is_complete = False và đưa các bước CHƯA LÀM vào remaining_steps (bỏ các bước đã làm xong)."""
)
replanner = replanner_prompt | llm.with_structured_output(ReplanDecision)


# ==========================================
# 4. ĐỊNH NGHĨA CÁC NODE TRONG LANGGRAPH
# ==========================================
def plan_node(state: PlanExecuteState):
    print("\n--- [PLANNER] Đang lập kế hoạch... ---")
    plan_res = planner.invoke({"input": state["input"]})
    for i, s in enumerate(plan_res.steps, 1):
        print(f"  Bước {i}: {s}")
    return {"plan": plan_res.steps}

def exec_node(state: PlanExecuteState):
    step = state["plan"][0]
    print(f"\n--- [EXECUTOR] Đang thực hiện: '{step}' ---")
    
    context = "\n".join([f"- {s}: {r}" for s, r in state.get("past_steps", [])])
    task_prompt = (
        f"Hãy sử dụng công cụ (tools) được cung cấp để thực hiện đúng nhiệm vụ sau: {step}\n"
        f"Trả lời ngắn gọn kết quả tìm được."
    )
    if context:
        task_prompt += f"\n\nThông tin đã có từ các bước trước:\n{context}"
        
    res = executor_agent.invoke({"messages": [("user", task_prompt)]})
    output = res["messages"][-1].content
    print(f"  -> Kết quả: {output}")
    return {"past_steps": [(step, output)]}

def replan_node(state: PlanExecuteState):
    print("\n--- [REPLANNER] Đang đánh giá lại tiến độ... ---")
    decision = replanner.invoke(state)
    
    if decision.is_complete and decision.final_response:
        print("  -> Đã hoàn thành mục tiêu!")
        return {"response": decision.final_response}
    
    next_steps = decision.remaining_steps
    # Fallback an toàn: nếu model quên điền remaining_steps, tự động cắt bỏ bước đầu vừa chạy
    if not next_steps:
        next_steps = state["plan"][1:]
        
    if not next_steps:
        # Nếu đã cạn sạch bước mà model chưa tổng hợp, tự ghép kết quả lại
        summary = "\n".join([f"{s}: {r}" for s, r in state["past_steps"]])
        return {"response": f"Tổng hợp kết quả:\n{summary}"}
        
    print(f"  -> Các bước tiếp theo: {next_steps}")
    return {"plan": next_steps}

def should_end(state: PlanExecuteState):
    if state.get("response"):
        return END
    return "executor"


# ==========================================
# 5. BUILD & CHẠY ĐỒ THỊ (STATE GRAPH)
# ==========================================
workflow = StateGraph(PlanExecuteState)

workflow.add_node("planner", plan_node)
workflow.add_node("executor", exec_node)
workflow.add_node("replanner", replan_node)

workflow.add_edge(START, "planner")
workflow.add_edge("planner", "executor")
workflow.add_edge("executor", "replanner")
workflow.add_conditional_edges("replanner", should_end, ["executor", END])

app = workflow.compile()

if __name__ == "__main__":
    config = {"recursion_limit": 15}
    inputs = {"input": "So sánh dân số Việt Nam và Thái Lan, tính độ chênh lệch cụ thể."}
    
    final_state = app.invoke(inputs, config=config)
    
    print("\n==========================================")
    print("KẾT QUẢ CUỐI CÙNG:")
    print(final_state.get("response"))


--- [PLANNER] Đang lập kế hoạch... ---
  Bước 1: 1. Tính toán dân số Việt Nam và Thái Lan
  Bước 2: 2. So sánh dân số hai nước này
  Bước 3: 3. Tính toán độ chênh lệch giữa dân số hai nước này
  Bước 4: 4. So sánh và phân tích kết quả

--- [EXECUTOR] Đang thực hiện: '1. Tính toán dân số Việt Nam và Thái Lan' ---
  -> Kết quả: Dân số Việt Nam là 100.3 triệu người, trong khi đó, dân số Thái Lan là 71.8 triệu người.

--- [REPLANNER] Đang đánh giá lại tiến độ... ---
  -> Đã hoàn thành mục tiêu!

KẾT QUẢ CUỐI CÙNG:
Dựa trên kết quả đã đạt được, độ chênh lệch giữa dân số Việt Nam và Thái Lan là 28.5 triệu người. Đây là một số liệu cụ thể và đã hoàn thành các bước trong kế hoạch.


In [7]:
import json
from pathlib import Path
from typing import List, Literal
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# ==========================================
# 1. KHỞI TẠO OLLAMA MODEL
# ==========================================
# Khởi tạo model Ollama local (qwen2.5 hoặc llama3.1)
llm = ChatOllama(model="qwen2.5:1.5b", temperature=0)


# ==========================================
# 2. SCHEMA DỮ LIỆU CÓ CẤU TRÚC (PYDANTIC)
# ==========================================
class EmailSummary(BaseModel):
    """Khuôn mẫu dữ liệu bóc tách từ nội dung email."""
    summary: str = Field(
        description="Tóm tắt ngắn gọn nội dung email trong 2-3 câu."
    )
    priority: Literal["cao", "trung bình", "thấp"] = Field(
        description="Độ ưu tiên của email: 'cao' (việc gấp, deadline sát), 'trung bình', hoặc 'thấp'."
    )
    action_items: List[str] = Field(
        default_factory=list,
        description="Danh sách các việc cần làm cụ thể trích ra từ email. Nếu không có việc gì cần làm, trả về danh sách rỗng."
    )


# ==========================================
# 3. TOOL GHI FILE JSON
# ==========================================
@tool
def save_summary_to_json(data: dict, filename: str = "email_summary.json") -> str:
    """Lưu dữ liệu tóm tắt email và danh sách công việc ra file JSON chuẩn UTF-8.
    
    Args:
        data: Dictionary chứa summary, priority và danh sách việc cần làm.
        filename: Tên file JSON cần lưu.
    """
    out_path = Path(filename)
    # indent=2 và ensure_ascii=False để hiển thị tiếng Việt có dấu chuẩn đẹp
    out_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    return str(out_path.resolve())


# ==========================================
# 4. LLM CHAIN TÓM TẮT EMAIL
# ==========================================
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Bạn là trợ lý đọc email thông minh. Hãy đọc nội dung email, tóm tắt ngắn gọn, "
        "đánh giá mức độ ưu tiên ('cao', 'trung bình', hoặc 'thấp'), và trích xuất danh sách các việc cần làm. "
        "Nếu không có việc cần làm, danh sách action_items để rỗng."
    ),
    ("human", "Nội dung email:\n\n{email_text}"),
])

# Nối Prompt với Model ép khuôn Pydantic (Structured Output)
summarize_chain = prompt | llm.with_structured_output(EmailSummary)


def summarize_single_email(email_text: str) -> EmailSummary:
    """Gọi chain tóm tắt cho một đoạn text email."""
    return summarize_chain.invoke({"email_text": email_text})


# ==========================================
# 5. XỬ LÝ NHIỀU EMAIL VÀ GỘP TODO TỔNG
# ==========================================
def process_email_inbox(inbox: List[str], output_file: str = "all_tasks_summary.json"):
    all_results = []
    master_todo_list = []

    print(f"Bắt đầu xử lý {len(inbox)} email trong hàng đợi...\n")

    for idx, email_content in enumerate(inbox, start=1):
        print(f"--- Đang xử lý Email #{idx} ---")
        try:
            # 1. Gọi chain bóc tách dữ liệu
            result: EmailSummary = summarize_single_email(email_content)
            
            # In thông tin ra console
            print(f"  • Độ ưu tiên: [{result.priority.upper()}]")
            print(f"  • Tóm tắt: {result.summary}")
            print(f"  • Số việc cần làm: {len(result.action_items)}")

            # 2. Gom dữ liệu kết quả
            email_record = {
                "email_id": idx,
                "priority": result.priority,
                "summary": result.summary,
                "action_items": result.action_items
            }
            all_results.append(email_record)

            # Gộp vào todo tổng với tiền tố rõ ràng
            for task in result.action_items:
                master_todo_list.append(f"[Email #{idx} - Ưu tiên {result.priority}] {task}")

        except Exception as e:
            # Bọc try/except để một email lỗi không làm dừng toàn bộ mẻ xử lý
            print(f"  ⚠️ Lỗi khi xử lý Email #{idx}: {e}. Bỏ qua email này.")

    # 3. Tạo cấu trúc file tổng hợp và lưu qua Tool
    final_payload = {
        "total_emails_processed": len(inbox),
        "successful": len(all_results),
        "master_todo_list": master_todo_list,
        "details": all_results
    }

    saved_path = save_summary_to_json.invoke({
        "data": final_payload,
        "filename": output_file
    })
    
    print("\n==========================================")
    print(f"✅ Hoàn tất! Đã lưu báo cáo và danh sách việc vào: {saved_path}")
    print(f"Tổng số việc cần làm gom được: {len(master_todo_list)} việc.")


# ==========================================
# TEST THỬ NGHIỆM VỚI DANH SÁCH EMAIL GIẢ LẬP
# ==========================================
if __name__ == "__main__":
    sample_emails = [
        # Email 1: Khẩn cấp
        """Chào Dương, báo cáo doanh thu Q3 phải chốt xong trước 17h thứ Sáu tuần này để gửi sếp tổng.
        Nhớ kiểm tra lại số liệu bảng cân đối kế toán trước khi gửi nhé!""",
        
        # Email 2: Bình thường / không gấp
        """Hi em, rảnh thì ghé qua thư viện trả quyển sách AI Agent hôm trước mượn nhé, không vội đâu.""",
        
        # Email 3: Thông báo thông tin (không có việc cần làm)
        """Kính gửi quý khách, hệ thống mạng nội bộ sẽ bảo trì định kỳ vào 23:00 tối nay. Trân trọng cảm ơn."""
    ]

    process_email_inbox(sample_emails)

Bắt đầu xử lý 3 email trong hàng đợi...

--- Đang xử lý Email #1 ---
  • Độ ưu tiên: [CAO]
  • Tóm tắt: Đã nhận được báo cáo doanh thu Q3 và yêu cầu hoàn thành trước 17h thứ Sáu tuần này. Cần kiểm tra lại số liệu bảng cân đối kế toán trước khi gửi.
  • Số việc cần làm: 2
--- Đang xử lý Email #2 ---
  • Độ ưu tiên: [CAO]
  • Tóm tắt: 朋友请求归还借阅的AI Agent书籍。
  • Số việc cần làm: 0
--- Đang xử lý Email #3 ---
  • Độ ưu tiên: [THẤP]
  • Tóm tắt: Hệ thống mạng nội bộ sẽ được bảo trì vào 23:00 tối nay. 
  • Số việc cần làm: 0

✅ Hoàn tất! Đã lưu báo cáo và danh sách việc vào: D:\Python project\AI_Agent\all_tasks_summary.json
Tổng số việc cần làm gom được: 2 việc.


In [13]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ==========================================
# 1. KHỞI TẠO OLLAMA LOCAL (Tối ưu tốc độ & context)
# ==========================================
llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0.1,
    num_ctx=2048,
    num_predict=300
)

# ==========================================
# 2. SCHEMA ÉP REVIEWER ĐƯA RA GÓP Ý RÕ RÀNG
# ==========================================
class ReviewDecision(BaseModel):
    is_approved: bool = Field(
        description="True nếu code đã chuẩn xác và đầy đủ kiểm tra biên. False nếu có lỗi logic hoặc thiếu xử lý biên."
    )
    issue_found: str = Field(
        description="Nếu is_approved=False, nêu rõ lỗi hoặc trường hợp kiểm thử bị thiếu (ví dụ: chưa bắt n không phải số nguyên, n âm). Nếu True, ghi 'Đã chuẩn'."
    )
    instruction_to_fix: str = Field(
        description="Nếu is_approved=False, hướng dẫn cụ thể cách sửa cho lập trình viên. Nếu True, ghi 'Không cần sửa'."
    )

reviewer_chain = llm.with_structured_output(ReviewDecision)

# ==========================================
# 3. STATE
# ==========================================
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    is_approved: bool
    review_feedback: str

# ==========================================
# 4. NODE WRITER
# ==========================================
def writer_node(state: AgentState):
    # Lấy thông tin phản hồi từ lượt trước nếu có
    feedback = state.get("review_feedback", "")
    
    if not feedback:
        user_prompt = f"Yêu cầu: {state['messages'][-1].content}\nHãy viết code Python hoàn chỉnh."
    else:
        user_prompt = f"Reviewer yêu cầu sửa lỗi sau:\n{feedback}\nHãy cập nhật và viết lại đoạn code Python hoàn chỉnh đã sửa lỗi."

    prompt = [
        ("system", "Bạn là lập trình viên Python. Chỉ cung cấp mã nguồn Python có chú thích ngắn gọn, giải quyết đúng yêu cầu."),
        ("human", user_prompt),
    ]

    response = llm.invoke(prompt)
    print("\n" + "=" * 45)
    print(f"👨‍💻 [Writer]:\n{response.content}")
    print("=" * 45)

    return {"messages": [AIMessage(content=response.content, name="Writer")]}

# ==========================================
# 5. NODE REVIEWER (Ép đưa ra lỗi và cách sửa cụ thể)
# ==========================================
def reviewer_node(state: AgentState):
    last_writer_code = state["messages"][-1].content

    prompt = [
        (
            "system",
            "Bạn là Senior QA Code Reviewer. Kiểm tra kỹ đoạn code Python của lập trình viên.\n"
            "Các tiêu chí bắt buộc:\n"
            "1. Có xử lý trường hợp n âm không?\n"
            "2. Có xử lý trường hợp n = 0 không?\n"
            "3. Có kiểm tra nếu n không phải số nguyên không?\n"
            "Nếu thỏa mãn TẤT CẢ tiêu chí -> is_approved = True.\n"
            "Nếu còn thiếu bất kỳ tiêu chí nào -> is_approved = False và nêu rõ lỗi vào issue_found cùng hướng sửa vào instruction_to_fix."
        ),
        ("human", f"Đoạn code cần review:\n\n{last_writer_code}"),
    ]

    decision: ReviewDecision = reviewer_chain.invoke(prompt)
    
    full_feedback = f"Lỗi: {decision.issue_found} | Hướng sửa: {decision.instruction_to_fix}"
    
    print("\n" + "=" * 45)
    print(f"🕵️ [Reviewer]")
    print(f"  • Duyệt: {decision.is_approved}")
    print(f"  • Lỗi tìm thấy: {decision.issue_found}")
    print(f"  • Yêu cầu sửa: {decision.instruction_to_fix}")
    print("=" * 45)

    return {
        "messages": [AIMessage(content=full_feedback, name="Reviewer")],
        "is_approved": decision.is_approved,
        "review_feedback": full_feedback
    }

# ==========================================
# 6. ĐIỀU KIỆN DỪNG
# ==========================================
def should_continue(state: AgentState):
    if state.get("is_approved", False):
        print("\n✅ [XÁC NHẬN] Code đã được duyệt hoàn chỉnh!")
        return END

    if len(state["messages"]) >= 10:
        print("\n⚠️ Đã đạt giới hạn số lần trao đổi (Max iterations). Dừng quy trình.")
        return END

    return "writer"

# ==========================================
# 7. BUILD & CHẠY
# ==========================================
workflow = StateGraph(AgentState)
workflow.add_node("writer", writer_node)
workflow.add_node("reviewer", reviewer_node)

workflow.add_edge(START, "writer")
workflow.add_edge("writer", "reviewer")
workflow.add_conditional_edges("reviewer", should_continue, ["writer", END])

app = workflow.compile()

if __name__ == "__main__":
    task = "Viết hàm tính giai thừa của n trong Python, có xử lý trường hợp n âm."
    inputs = {
        "messages": [HumanMessage(content=task)], 
        "is_approved": False,
        "review_feedback": ""
    }
    app.invoke(inputs)


👨‍💻 [Writer]:
```python
def factorial(n):
    """
    Hàm tính giai thừa của n
    :param n: Số nguyên dương
    :return: Giai thừa của n
    """
    if n < 0:
        return "Không thể tính giai thừa của số âm"
    elif n == 0:
        return 1
    else:
        result = 1
        for i in range(1, n + 1):
            result *= i
        return result

# Kiểm tra hàm
print(factorial(5))  # Expected output: 120
print(factorial(0))  # Expected output: 1
print(factorial(-1)) # Expected output: Không thể tính giai thừa của số âm
```

🕵️ [Reviewer]
  • Duyệt: False
  • Lỗi tìm thấy: Hàm không kiểm tra nếu n không phải số nguyên
  • Yêu cầu sửa: Hàm cần kiểm tra nếu n không phải số nguyên trước khi thực hiện các phép tính.

👨‍💻 [Writer]:
Đoạn mã Python sau đây có lỗi và cần được cập nhật để kiểm tra nếu số n không phải là số nguyên:

```python
def tính_hệ_lượng(n):
    if not isinstance(n, int):
        raise TypeError("n không phải là số nguyên")
    return n ** 2

# Sử dụng hàm
n = 5
pri

# Luồng RAG với LangGraph (Local)

RAG (Retrieval-Augmented Generation) gồm 2 việc: **tìm đoạn tài liệu liên quan** rồi **đưa cho LLM trả lời dựa trên đó**. LLM không tự biết tài liệu nội bộ, nên cần bước truy xuất trước.

## 1. Tổng quan

```text
GIAI ĐOẠN 1 - INDEXING (làm một lần)

  Tài liệu gốc ──> Cắt thành chunk ──> Embedding (chunk -> vector) ──> ChromaDB
                                                                          │
                                                                          │ tìm kiếm
GIAI ĐOẠN 2 - HỎI ĐÁP (LangGraph)                                         │
                                                                          ▼
  START ──> retrieve ──> grade ──┬── có đoạn liên quan ──> generate ──> END
                                 │
                                 └── không có đoạn nào ──> no_answer ──> END
```

## 2. Giai đoạn Indexing

| Bước | Việc làm | Công cụ | Mục đích |
|------|----------|---------|----------|
| 1 | Đọc tài liệu thành `Document` | `langchain_core` | Gắn nội dung và metadata (nguồn) |
| 2 | Cắt chunk (300 ký tự, gối 30) | `RecursiveCharacterTextSplitter` | Mỗi chunk chứa một ý, tìm chính xác hơn |
| 3 | Embedding | `OllamaEmbeddings` | Biến chunk thành vector, nghĩa gần thì vector gần |
| 4 | Lưu trữ | `Chroma` (cosine) | Cho phép tìm theo độ tương đồng |

## 3. State (dữ liệu đi xuyên đồ thị)

```python
class RAGState(TypedDict):
    question: str             # câu hỏi
    context: List[Document]   # các đoạn được giữ lại
    distances: List[float]    # khoảng cách tương ứng
    answer: str               # câu trả lời cuối
```

## 4. Các node

| Node | Đọc từ state | Ghi vào state | Việc làm |
|------|--------------|---------------|----------|
| `retrieve` | `question` | `context`, `distances` | Embed câu hỏi, lấy `TOP_K` chunk gần nhất |
| `grade` | `context`, `distances` | `context` | Bỏ đoạn có `distance > MAX_DISTANCE` |
| `generate` | `question`, `context` | `answer` | Ghép ngữ cảnh, gọi LLM với system prompt chống bịa |
| `no_answer` | không | `answer` | Trả "Tài liệu không đề cập", không gọi LLM |

## 5. Điều kiện rẽ nhánh

```python
def route_after_grade(state):
    return "generate" if state["context"] else "no_answer"
```

- Còn ít nhất một đoạn sau khi lọc: đi `generate`.
- Không còn đoạn nào: đi `no_answer`.

## 6. Ví dụ chạy

**Câu hỏi có trong tài liệu:** "Tôi được WFH mấy ngày một tuần?"

1. `retrieve` lấy 3 đoạn, đoạn về WFH có distance thấp nhất.
2. `grade` giữ lại đoạn đó (và có thể vài đoạn khác).
3. `generate` gửi ngữ cảnh cho qwen, qwen trả lời "tối đa 2 ngày, Thứ 4 và Thứ 6, nguồn so_tay_nhan_su.txt".

**Câu hỏi ngoài tài liệu:** "Công ty có thưởng Tết bằng cổ phiếu không?"

1. `retrieve` vẫn trả 3 đoạn, nhưng đều có distance cao.
2. `grade` loại hết.
3. Đi `no_answer`, trả lời ngay "Tài liệu không đề cập đến nội dung này."

## 7. Các tham số cần tinh chỉnh

| Tham số | Mặc định | Tăng thì | Giảm thì |
|---------|----------|----------|----------|
| `TOP_K` | 3 | Nhiều ngữ cảnh hơn, dễ lẫn nhiễu | Có thể thiếu thông tin |
| `MAX_DISTANCE` | 0.55 | Dễ chấp nhận đoạn không liên quan | Dễ loại nhầm đoạn đúng |
| `chunk_size` | 300 | Mỗi đoạn chứa nhiều ý hơn | Tìm chính xác hơn nhưng mất ngữ cảnh |
| `EMBED_MODEL` | nomic-embed-text | Đổi sang `bge-m3` để tiếng Việt tốt hơn | |

## 8. Vì sao cần bước `grade`?

Retriever luôn trả về `TOP_K` đoạn dù câu hỏi không liên quan. Nếu đưa nguyên cho LLM nhỏ, nó dễ bịa câu trả lời từ ngữ cảnh sai. Lọc theo ngưỡng và rẽ nhánh `no_answer` là cách chống hallucination đơn giản nhất.

In [1]:
from typing import TypedDict, List

from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langgraph.graph import StateGraph, START, END


In [5]:
# khai bao model
LLM_MODEL = "qwen2.5:1.5b" 
EMBED_MODEL = "nomic-embed-text:latest"
TOP_K = 3 
MAX_DISTANCE = 0.55 

In [10]:
# ep khuon model 
embeddings = OllamaEmbeddings(model= EMBED_MODEL)
llm = ChatOllama(model= LLM_MODEL, temperature= 0)

In [7]:
# Dữ liệu mẫu. Thực tế bạn sẽ đọc từ file .txt/.pdf rồi tạo Document tương tự.
raw_documents = [
    Document(
        page_content="Chính sách nghỉ phép năm 2026: Nhân viên chính thức có 15 ngày phép nguyên lương mỗi năm. "
                     "Làm thêm mỗi 3 năm được cộng 1 ngày.",
        metadata={"source": "so_tay_nhan_su.txt"},
    ),
    Document(
        page_content="Quy định làm việc từ xa (Work From Home): Nhân viên được WFH tối đa 2 ngày/tuần "
                     "vào Thứ 4 và Thứ 6, cần báo trước trên hệ thống trước 9h sáng.",
        metadata={"source": "so_tay_nhan_su.txt"},
    ),
    Document(
        page_content="Hỗ trợ thiết bị: Công ty cấp ngân sách 5.000.000 VNĐ/năm để nhân viên mua sắm chuột, "
                     "bàn phím, hoặc nâng cấp RAM/SSD phục vụ công việc.",
        metadata={"source": "chinh_sach_it.txt"},
    ),
    Document(
        page_content="Giờ làm việc: Từ Thứ 2 đến Thứ 6, 8h30 đến 17h30, nghỉ trưa 1 tiếng từ 12h đến 13h. "
                     "Đi trễ quá 3 lần/tháng sẽ bị nhắc nhở.",
        metadata={"source": "so_tay_nhan_su.txt"},
    ),
]

In [8]:
# chunking van ban => chia nho => vector hoa de dang hon 
splitter = RecursiveCharacterTextSplitter(chunk_size= 300, chunk_overlap= 30) #chunk_overlap: các đoạn gối nhau một chút để không cắt mất ngữ cảnh ở ranh giới
chunks = splitter.split_documents(raw_documents )

In [11]:
# bien chunk thanh vector 
vectorstore = Chroma.from_documents(
    documents= chunks, 
    embedding= embeddings, 
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory= "./chroma_db",
)
print(f"[INDEX] {len(raw_documents)} tài liệu -> {len(chunks)} chunk đã được lưu vào ChromaDB\n")


[INDEX] 4 tài liệu -> 4 chunk đã được lưu vào ChromaDB



In [18]:
# state =>  trang thai di xuyen suot do thi => 1 node 1 state => tra ve phan state muon cap nhat 
class RAGState(TypedDict) :
    question: str  # cau hoi nguo dung 
    context: List[Document] # cac doan tai lieu duoc giu lai sau khi loc 
    distances: List[float] # khoang cach tuong ung 
    answer: str # cau tra loi cuoi cungh 

In [19]:
# cac node cua lang graph 
def retrieve_node(state: RAGState) : 
    """Node 1 - RETRIEVAL: embed câu hỏi rồi tìm TOP_K chunk gần nhất."""
    print("--- 🔍 BƯỚC 1: TRUY XUẤT ---")
    results = vectorstore.similarity_search_with_score(state["question"], k=TOP_K)

    for doc, dist in results:
        print(f"    distance={dist:.3f} | {doc.metadata['source']} | {doc.page_content[:60]}...")
    return {
        "context": [doc for doc, _ in results],
        "distances": [dist for _, dist in results],
    }

def grade_node(state: RAGState) :
    """Node 2 - LỌC: bỏ những đoạn quá xa câu hỏi.
    
        Nếu không có bước này, retriever LUÔN trả về TOP_K đoạn dù chẳng liên quan,
        và LLM nhỏ rất dễ bịa ra câu trả lời từ đống ngữ cảnh sai đó.
    """
    print("--- ⚖️  BƯỚC 2: LỌC THEO NGƯỠNG ---")
    kept = [
        doc for doc, dist in zip(state["context"], state["distances"])
        if dist <= MAX_DISTANCE
    ]
    print(f"    Giữ lại {len(kept)}/{len(state['context'])} đoạn (ngưỡng {MAX_DISTANCE})")
    return {"context": kept}

def generate_node(state: RAGState) :
    """Node 3 - GENERATION: đưa ngữ cảnh + câu hỏi cho LLM."""
    # ghep doan ngu canh + nguon LLM 
    context_text = "\n\n".join(
        f"[Nguồn: {d.metadata.get('source', 'N/A')}] {d.page_content}"
        for d in state["context"]
    )
    # system msg 
    messages = [
        SystemMessage(content=(
            "Bạn là trợ lý nội bộ của công ty. Chỉ dựa vào NGỮ CẢNH được cung cấp để trả lời. "
            "Nếu ngữ cảnh không có thông tin, hãy nói rõ 'Tài liệu không đề cập', tuyệt đối không tự bịa. "
            "Trả lời ngắn gọn bằng tiếng Việt và nêu tên nguồn."
        )),
        HumanMessage(content=f"NGỮ CẢNH:\n{context_text}\n\nCÂU HỎI: {state['question']}"),
    ]
    response = llm.invoke(messages)
    return {"answer": response.content} 

def no_answer_node(state: RAGState):
    """Node dự phòng: không có đoạn nào liên quan -> trả lời luôn, KHÔNG gọi LLM."""
    print("--- 🚫 KHÔNG TÌM THẤY TÀI LIỆU LIÊN QUAN ---")
    return {"answer": "Tài liệu không đề cập đến nội dung này."}

def route_after_grade(state: RAGState) -> str : 
    return "generate" if state["context"] else "no_answer"


In [20]:
# rap do thi 
workflow = StateGraph(RAGState) 

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("generate", generate_node)
workflow.add_node("no_answer", no_answer_node)


workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    route_after_grade,
    {"generate": "generate", "no_answer": "no_answer"},
)
workflow.add_edge("generate", END)
workflow.add_edge("no_answer", END)

app = workflow.compile()


In [21]:
def ask(question: str):
    print(f"\n❓ Hỏi: {question}")
    result = app.invoke({"question": question})
    print(f"✅ Trả lời:\n{result['answer']}")
    print("\n" + "=" * 60)


if __name__ == "__main__":
    # Có trong tài liệu
    ask("Tôi được làm ở nhà mấy ngày 1 tuần và vào thứ mấy?")
    ask("Công ty hỗ trợ bao nhiêu tiền để mua bàn phím?")

    # Ngoài tài liệu -> kỳ vọng đi nhánh no_answer
    ask("Công ty có thưởng Tết bằng cổ phiếu không?")


❓ Hỏi: Tôi được làm ở nhà mấy ngày 1 tuần và vào thứ mấy?
--- 🔍 BƯỚC 1: TRUY XUẤT ---
    distance=0.245 | so_tay_nhan_su.txt | Giờ làm việc: Từ Thứ 2 đến Thứ 6, 8h30 đến 17h30, nghỉ trưa ...
    distance=0.278 | so_tay_nhan_su.txt | Quy định làm việc từ xa (Work From Home): Nhân viên được WFH...
    distance=0.290 | so_tay_nhan_su.txt | Chính sách nghỉ phép năm 2026: Nhân viên chính thức có 15 ng...
--- ⚖️  BƯỚC 2: LỌC THEO NGƯỠNG ---
    Giữ lại 3/3 đoạn (ngưỡng 0.55)
✅ Trả lời:
Theo ngữ cảnh, bạn được phép làm việc từ xa (Work From Home) tối đa 2 ngày/tuần vào Thứ 4 và Thứ 6.


❓ Hỏi: Công ty hỗ trợ bao nhiêu tiền để mua bàn phím?
--- 🔍 BƯỚC 1: TRUY XUẤT ---
    distance=0.248 | chinh_sach_it.txt | Hỗ trợ thiết bị: Công ty cấp ngân sách 5.000.000 VNĐ/năm để ...
    distance=0.289 | so_tay_nhan_su.txt | Giờ làm việc: Từ Thứ 2 đến Thứ 6, 8h30 đến 17h30, nghỉ trưa ...
    distance=0.304 | so_tay_nhan_su.txt | Quy định làm việc từ xa (Work From Home): Nhân viên được WFH...
--- ⚖️  BƯỚC